# MESA Adaptive Strategy — MAMA/FAMA Crossover

**Hypothesis:** The MESA Adaptive Moving Average (MAMA) adapts to the market's dominant cycle period via Hilbert Transform phase measurement. MAMA/FAMA crossovers capture cycle turns with less whipsaw than fixed-period MA crossovers because the smoothing factor self-adjusts.

**Core Idea (Ehlers):** Price exhibits dominant cycles of varying period. Fixed MAs are always wrong — too fast in trends, too slow in cycles. MAMA solves this by reading the instantaneous frequency and adapting alpha between `fast_limit` (0.5) and `slow_limit` (0.05).

**Asset:** SPY (S&P 500 ETF)  
**Timeframe:** Daily  
**Data Source:** yfinance  
**Indicators:** Custom Ehlers DSP (lib/ehlers_dsp.py)  
**Backtest Engine:** vectorbt  

**Alpha Factory Context:**
- Source: Seed query "MESA adaptive moving average Ehlers"
- Variant ID: `mesa-mama-fama-spy-daily-v1`
- Related: `mesa-mama-fama-spy-daily-v2` (with regime filter)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikolas-joyce/swarm-alpha-notebooks/blob/main/strategies/mesa_adaptive.ipynb)

## 0. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q vectorbt==0.26.2 yfinance>=0.2.36
    !git clone --depth 1 https://github.com/nikolas-joyce/swarm-alpha-notebooks.git /tmp/swarm-alpha 2>/dev/null || true
    sys.path.insert(0, '/tmp/swarm-alpha')
else:
    sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from lib.ehlers_dsp import (
    mama_fama, hilbert_transform_dominant_cycle, instantaneous_trendline,
    cyber_cycle, adaptive_rsi, roofing_filter, compute_ehlers_suite
)
from lib.qc_gate import apply_qc_gate, export_qc_result, QC_THRESHOLDS
from lib.backtest_utils import (
    compute_metrics, merge_is_oos_metrics, print_comparison
)

print('All imports OK')
print(f'QC thresholds: {QC_THRESHOLDS}')

## 1. Data Download

In [ ]:
# ── Parameters ────────────────────────────────────────────────────
TICKER = "SPY"
START_DATE = "2010-01-01"  # Longer history for cycle detection
OOS_SPLIT = 0.7
INITIAL_CASH = 100_000

# MAMA/FAMA parameters
FAST_LIMIT = 0.5   # Max alpha (fast tracking)
SLOW_LIMIT = 0.05  # Min alpha (slow tracking)

# ── Download ──────────────────────────────────────────────────────
raw = yf.download(TICKER, start=START_DATE, auto_adjust=True)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
print(f"Downloaded {len(df)} bars: {df.index[0].date()} → {df.index[-1].date()}")

## 2. Ehlers Indicator Suite

In [ ]:
# ── Compute Ehlers indicators ────────────────────────────────────
# Use (H+L)/2 as input — Ehlers recommends this over Close
hl2 = (df["High"] + df["Low"]) / 2

# Core MAMA/FAMA
df["MAMA"], df["FAMA"] = mama_fama(hl2, FAST_LIMIT, SLOW_LIMIT)

# Dominant cycle period
df["dominant_cycle"] = hilbert_transform_dominant_cycle(hl2)

# Supporting indicators
df["iTrend"] = instantaneous_trendline(hl2)
df["cyber_cycle"] = cyber_cycle(hl2)
df["adaptive_rsi"] = adaptive_rsi(hl2)
df["roofing"] = roofing_filter(hl2)

# Warmup: drop first 50 bars (Hilbert needs history)
df = df.iloc[50:].copy()

print(f"Indicators computed: {len(df)} bars after warmup")
print(f"Dominant cycle range: {df['dominant_cycle'].min():.1f} — {df['dominant_cycle'].max():.1f}")
print(f"Dominant cycle mean: {df['dominant_cycle'].mean():.1f} bars")
df[["Close", "MAMA", "FAMA", "dominant_cycle"]].tail(10)

## 3. Signal Generation — MAMA/FAMA Crossover

**Base signal:** MAMA crosses above FAMA → long. MAMA crosses below FAMA → exit.

We test three variants:
1. **v1: Pure crossover** — no filter
2. **v2: Trend filter** — only trade when Close > iTrend (uptrend)
3. **v3: Cycle + trend filter** — add cyber_cycle confirmation

In [ ]:
# ── Variant 1: Pure MAMA/FAMA crossover ──────────────────────────
mama_above = df["MAMA"] > df["FAMA"]
entries_v1 = mama_above & ~mama_above.shift(1).fillna(False)  # cross above
exits_v1 = ~mama_above & mama_above.shift(1).fillna(True)    # cross below

# ── Variant 2: + Trend filter (Close > iTrend) ───────────────────
trend_up = df["Close"] > df["iTrend"]
entries_v2 = entries_v1 & trend_up
exits_v2 = exits_v1  # exit on any cross-down regardless of trend

# ── Variant 3: + Cyber cycle confirmation ────────────────────────
cycle_up = df["cyber_cycle"] > 0
entries_v3 = entries_v1 & trend_up & cycle_up
exits_v3 = exits_v1 | (~cycle_up & ~trend_up)  # exit on cross-down OR both filters negative

print(f"V1 (pure crossover):   {entries_v1.sum()} entries")
print(f"V2 (+ trend filter):   {entries_v2.sum()} entries")
print(f"V3 (+ cycle + trend):  {entries_v3.sum()} entries")

## 4. IS/OOS Backtest — All Three Variants

In [ ]:
split_idx = int(len(df) * OOS_SPLIT)
split_date = df.index[split_idx]
print(f"IS: {df.index[0].date()} → {split_date.date()} ({split_idx} bars)")
print(f"OOS: {split_date.date()} → {df.index[-1].date()} ({len(df) - split_idx} bars)")

variants = {
    "v1_pure": (entries_v1, exits_v1),
    "v2_trend": (entries_v2, exits_v2),
    "v3_cycle_trend": (entries_v3, exits_v3),
}

all_results = {}

for name, (ent, ext) in variants.items():
    print(f"\n{'='*60}")
    print(f"  Variant: {name}")
    print(f"{'='*60}")

    # IS
    pf_is = vbt.Portfolio.from_signals(
        df["Close"].iloc[:split_idx],
        ent.iloc[:split_idx],
        ext.iloc[:split_idx],
        init_cash=INITIAL_CASH, freq="1D"
    )
    # OOS
    pf_oos = vbt.Portfolio.from_signals(
        df["Close"].iloc[split_idx:],
        ent.iloc[split_idx:],
        ext.iloc[split_idx:],
        init_cash=INITIAL_CASH, freq="1D"
    )

    is_m = compute_metrics(pf_is, f"{name} IS")
    oos_m = compute_metrics(pf_oos, f"{name} OOS")
    print_comparison(is_m, oos_m)

    qc_input = merge_is_oos_metrics(is_m, oos_m)
    passed, failures = apply_qc_gate(qc_input)

    print(f"\nQC Gate: {'PASS' if passed else 'FAIL'}")
    for f in failures:
        print(f"  ✗ {f}")

    all_results[name] = {
        "is_metrics": is_m,
        "oos_metrics": oos_m,
        "qc_input": qc_input,
        "passed": passed,
        "failures": failures,
        "pf_is": pf_is,
        "pf_oos": pf_oos,
    }

## 5. Variant Comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────────
summary_rows = []
for name, r in all_results.items():
    q = r["qc_input"]
    summary_rows.append({
        "variant": name,
        "sharpe_is": q["sharpe_is"],
        "sharpe_oos": q["sharpe_oos"],
        "max_dd": q["max_drawdown"],
        "calmar": q["calmar_ratio"],
        "trades": q["total_trades"],
        "qc_pass": r["passed"],
    })

summary = pd.DataFrame(summary_rows).set_index("variant")
print("\nVariant Comparison:")
print(summary.to_string())

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

# Panel 1: Price + MAMA + FAMA
axes[0].plot(df.index, df["Close"], label="Close", linewidth=0.6, alpha=0.7)
axes[0].plot(df.index, df["MAMA"], label="MAMA", linewidth=1.0, color="blue")
axes[0].plot(df.index, df["FAMA"], label="FAMA", linewidth=1.0, color="red")
axes[0].plot(df.index, df["iTrend"], label="iTrend", linewidth=0.8, color="green", linestyle="--")
axes[0].axvline(split_date, color="black", linestyle="--", alpha=0.5, label="IS/OOS")
axes[0].set_title(f"{TICKER} — MESA Adaptive (MAMA/FAMA)")
axes[0].legend(loc="upper left", fontsize=8)
axes[0].set_ylabel("Price")

# Panel 2: Dominant Cycle Period
axes[1].plot(df.index, df["dominant_cycle"], linewidth=0.8, color="purple")
axes[1].axhline(20, color="gray", linestyle=":", alpha=0.5, label="20-bar ref")
axes[1].axvline(split_date, color="black", linestyle="--", alpha=0.5)
axes[1].set_ylabel("Dominant Cycle\n(bars)")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_ylim(5, 52)

# Panel 3: Cyber Cycle
axes[2].plot(df.index, df["cyber_cycle"], linewidth=0.7, color="teal")
axes[2].axhline(0, color="gray", linestyle="-", alpha=0.3)
axes[2].axvline(split_date, color="black", linestyle="--", alpha=0.5)
axes[2].set_ylabel("Cyber Cycle")

# Panel 4: Adaptive RSI
axes[3].plot(df.index, df["adaptive_rsi"], linewidth=0.7, color="orange")
axes[3].axhline(70, color="red", linestyle="--", alpha=0.4)
axes[3].axhline(30, color="green", linestyle="--", alpha=0.4)
axes[3].axvline(split_date, color="black", linestyle="--", alpha=0.5)
axes[3].set_ylabel("Adaptive RSI")
axes[3].set_ylim(0, 100)

# Panel 5: Equity curves (all 3 variants)
colors = {"v1_pure": "steelblue", "v2_trend": "forestgreen", "v3_cycle_trend": "darkred"}
for name, (ent, ext) in variants.items():
    pf = vbt.Portfolio.from_signals(
        df["Close"], ent, ext, init_cash=INITIAL_CASH, freq="1D"
    )
    eq = pf.value()
    axes[4].plot(eq.index, eq.values, label=name, linewidth=0.8, color=colors[name])

axes[4].axvline(split_date, color="black", linestyle="--", alpha=0.5)
axes[4].axhline(INITIAL_CASH, color="gray", linestyle=":", alpha=0.5)
axes[4].set_ylabel("Portfolio ($)")
axes[4].legend(loc="upper left", fontsize=8)
axes[4].set_xlabel("Date")

plt.tight_layout()
plt.show()

## 7. Dominant Cycle Analysis

Examine how the dominant cycle period varies across market regimes.

In [ ]:
# ── Cycle period distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(df["dominant_cycle"], bins=30, color="purple", alpha=0.7, edgecolor="black")
axes[0].axvline(df["dominant_cycle"].mean(), color="red", linestyle="--", label=f'Mean: {df["dominant_cycle"].mean():.1f}')
axes[0].axvline(df["dominant_cycle"].median(), color="blue", linestyle="--", label=f'Median: {df["dominant_cycle"].median():.1f}')
axes[0].set_xlabel("Dominant Cycle Period (bars)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Cycle Period Distribution")
axes[0].legend()

# Rolling stats
dc_roll = df["dominant_cycle"].rolling(63)  # ~3 months
axes[1].plot(df.index, dc_roll.mean(), label="63-day mean", color="purple")
axes[1].fill_between(
    df.index,
    dc_roll.mean() - dc_roll.std(),
    dc_roll.mean() + dc_roll.std(),
    alpha=0.2, color="purple"
)
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Cycle Period (bars)")
axes[1].set_title("Rolling Dominant Cycle (63-day)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nCycle period stats:")
print(f"  Mean: {df['dominant_cycle'].mean():.1f} bars")
print(f"  Median: {df['dominant_cycle'].median():.1f} bars")
print(f"  Std: {df['dominant_cycle'].std():.1f} bars")
print(f"  Range: [{df['dominant_cycle'].min():.1f}, {df['dominant_cycle'].max():.1f}]")

## 8. Parameter Sensitivity — MAMA Fast/Slow Limits

In [ ]:
# ── Sweep fast_limit × slow_limit on IS data ────────────────────
fast_limits = [0.3, 0.4, 0.5, 0.6, 0.7]
slow_limits = [0.01, 0.03, 0.05, 0.08, 0.1]

sweep_results = []
price_is = df["Close"].iloc[:split_idx]
hl2_is = ((df["High"] + df["Low"]) / 2).iloc[:split_idx]

for fl in fast_limits:
    for sl in slow_limits:
        if sl >= fl:
            continue  # slow must be < fast
        try:
            m, f = mama_fama(hl2_is, fast_limit=fl, slow_limit=sl)
            above = m > f
            ent = above & ~above.shift(1).fillna(False)
            ext = ~above & above.shift(1).fillna(True)

            pf = vbt.Portfolio.from_signals(
                price_is, ent, ext, init_cash=INITIAL_CASH, freq="1D"
            )
            stats = pf.stats()
            sweep_results.append({
                "fast": fl,
                "slow": sl,
                "sharpe": float(stats.get("Sharpe Ratio", 0)),
                "max_dd": abs(float(stats.get("Max Drawdown [%]", 100))) / 100,
                "trades": int(stats.get("Total Trades", 0)),
            })
        except Exception:
            pass

sweep_df = pd.DataFrame(sweep_results)
if len(sweep_df) > 0:
    pivot = sweep_df.pivot_table(values="sharpe", index="slow", columns="fast")
    print("Sharpe Ratio Heatmap (IS, MAMA/FAMA crossover):")
    print(pivot.round(3).to_string())
    print(f"\nBest: {sweep_df.loc[sweep_df['sharpe'].idxmax()].to_dict()}")
else:
    print("No valid sweep results")

## 9. Export Best Variant for Alpha Factory

In [ ]:
# ── Pick best variant and export ─────────────────────────────────
# Select the variant with highest OOS Sharpe that passed QC
# If none passed, export the best anyway (goes to graveyard)

best_name = None
best_sharpe_oos = -999

for name, r in all_results.items():
    oos_sharpe = r["qc_input"]["sharpe_oos"]
    if r["passed"] and oos_sharpe > best_sharpe_oos:
        best_name = name
        best_sharpe_oos = oos_sharpe

# If nothing passed, take best OOS Sharpe anyway
if best_name is None:
    for name, r in all_results.items():
        oos_sharpe = r["qc_input"]["sharpe_oos"]
        if oos_sharpe > best_sharpe_oos:
            best_name = name
            best_sharpe_oos = oos_sharpe

best = all_results[best_name]
print(f"Exporting: {best_name} (OOS Sharpe: {best_sharpe_oos:.3f})")

result_path = export_qc_result(
    strategy_name=f"MESA MAMA/FAMA ({best_name}) — {TICKER} Daily",
    variant_id=f"mesa-mama-fama-spy-daily-{best_name}",
    results=best["qc_input"],
    passed=best["passed"],
    failures=best["failures"],
    output_dir="../results" if not IN_COLAB else "/content/results",
)

print(f"\nExported to: {result_path}")
print("\nCopy to 12-Alpha-Factory/data/results/ then run Station 5")

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(result_path))

---

## Notes & Next Variants

**What this tests:**
- Ehlers' core thesis: adaptive smoothing via Hilbert Transform phase measurement outperforms fixed-period MAs
- Three filter combinations to find the right signal-to-noise tradeoff
- Dominant cycle as a market regime descriptor

**Known limitations:**
- Hilbert Transform needs ~50 bars warmup — first entries are unreliable
- Daily bars may be too coarse for cycle detection (Ehlers originally designed for intraday)
- Long-only; short leg on MAMA < FAMA could capture downswing alpha
- No vol scaling — position size is fixed

**Potential next variants:**
1. **v4: Adaptive RSI confirmation** — only enter when adaptive_rsi < 40 (oversold on cycle basis)
2. **v5: Multi-asset** — run MAMA/FAMA on QQQ, IWM, TLT, GLD for diversification
3. **v6: Intraday** — use 15-min bars from yfinance (60-day history) to test Ehlers' intended timeframe
4. **v7: Roofing filter pre-processing** — feed roofing-filtered price into MAMA instead of raw HL2
5. **v8: Dominant cycle regime switch** — mean-reversion when DC < 15, trend-following when DC > 30